# Auto Insurance Customer Value & Renewal Offer Intelligence

## Business problem
The insurer spends money on renewal offers and acquisition channels without a clear view of which customer segments are actually profitable (value generated vs. claims cost) or which segments respond to which offers. Decisions on where to focus retention spend and which offers/channels to prioritize are currently not grounded in segment-level profitability or response data.

## Analytical objective:
Build a Customer Value & Offer Response Intelligence analysis that segments customers by relative profitability (CLV vs. claims cost) and by renewal-offer response, to identify which customer profiles are high-value and offer-responsive vs. high-cost and unresponsive — informing where retention and acquisition spend should be targeted.

## Dataset
`AutoInsurance.csv` — 9,134 customers, single snapshot (2011),
5 U.S. states (CA, OR, AZ, NV, WA).

## Approach 
Inspect → Clean (justified) → Reusable functions → Descriptive → Diagnostic → Exploratory, with every finding challenged for alternative explanations before being accepted as a conclusion.

In [6]:
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("Setup complete.")

Setup complete.


In [7]:
import os
os.environ.pop("MPLBACKEND", None)

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt

## Part 1 — Inspect the raw data before touching it

In [8]:
df_raw = pd.read_csv("/Users/mac/Desktop/Auto-Insurance-Customer-Value-Renewal-Offer-Intelligence/Dataset/AutoInsurance.csv")

print("Shape:", df_raw.shape)
print("\nDtypes:\n", df_raw.dtypes)
print("\nMissing values total:", df_raw.isna().sum().sum())
print("Duplicate rows:", df_raw.duplicated().sum())
print("Duplicate Customer IDs:", df_raw.duplicated(subset=['Customer']).sum())

Shape: (9134, 24)

Dtypes:
 Customer                          object
State                             object
Customer Lifetime Value          float64
Response                          object
Coverage                          object
Education                         object
Effective To Date                 object
EmploymentStatus                  object
Gender                            object
Income                             int64
Location Code                     object
Marital Status                    object
Monthly Premium Auto               int64
Months Since Last Claim            int64
Months Since Policy Inception      int64
Number of Open Complaints          int64
Number of Policies                 int64
Policy Type                       object
Policy                            object
Renew Offer Type                  object
Sales Channel                     object
Total Claim Amount               float64
Vehicle Class                     object
Vehicle Size                 

**What this tells us:** no missing values, no duplicates, 9,134 rows = 9,134
unique customers (one row per customer — a single snapshot, not a time series).

**What this does NOT tell us yet:** whether values like `Income = 0` are real
or a missing-data placeholder, or whether `Policy` and `Policy Type` are
redundant. Checked next.

In [9]:
zero_income = df_raw[df_raw['Income'] == 0]
print(f"Rows with Income = 0: {len(zero_income)} ({len(zero_income)/len(df_raw)*100:.1f}%)")
print("\nEmploymentStatus breakdown WITHIN Income=0 rows:")
print(zero_income['EmploymentStatus'].value_counts())
print("\nCross-tab: EmploymentStatus vs (Income==0)")
print(pd.crosstab(df_raw['EmploymentStatus'], df_raw['Income']==0))

print("\n--- Policy vs Policy Type redundancy check ---")
print(pd.crosstab(df_raw['Policy Type'], df_raw['Policy']))

print("\n--- Outlier check (IQR method) ---")
for col in ['Customer Lifetime Value', 'Monthly Premium Auto', 'Total Claim Amount', 'Income']:
    q1, q3 = df_raw[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = df_raw[(df_raw[col] < lower) | (df_raw[col] > upper)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df_raw)*100:.1f}%), max={df_raw[col].max():,.0f}")

print("\nTop 5 Monthly Premium Auto rows (checking if outliers are real):")
print(df_raw.nlargest(5, 'Monthly Premium Auto')[['Monthly Premium Auto','Coverage','Vehicle Class']])

Rows with Income = 0: 2317 (25.4%)

EmploymentStatus breakdown WITHIN Income=0 rows:
EmploymentStatus
Unemployed    2317
Name: count, dtype: int64

Cross-tab: EmploymentStatus vs (Income==0)


Income            False  True 
EmploymentStatus              
Disabled            405      0
Employed           5698      0
Medical Leave       432      0
Retired             282      0
Unemployed            0   2317

--- Policy vs Policy Type redundancy check ---
Policy          Corporate L1  Corporate L2  Corporate L3  Personal L1  Personal L2  Personal L3  Special L1  Special L2  Special L3
Policy Type                                                                                                                        
Corporate Auto           359           595          1014            0            0            0           0           0           0
Personal Auto              0             0             0         1240         2122         3426           0           0           0
Special Auto               0             0             0            0            0            0          66         164         148

--- Outlier check (IQR method) ---
Customer Lifetime Value: 817 outliers (

**What this tells us:**
- `Income = 0` maps **100% exactly** to `EmploymentStatus = Unemployed` (2,317
  rows) — a legitimate pattern, not missing data.
- `Policy` is a strict hierarchy of `Policy Type` + tier suffix (L1/L2/L3),
  confirmed by zero overlap in the cross-tab — treat as a drill-down, not an
  independent dimension.
- Outliers in CLV (8.9%), Premium (4.7%), and Claim Amount (5.0%) exist, but
  the highest premiums are consistently Premium-coverage Luxury vehicles —
  a real-world pattern, not a data error.

**What this does NOT tell us:** whether the data-generation process behind
CLV and Claim Amount is fully organic — that's investigated in Dx2 and Ex3
below.

## Part 2 — Clean and transform (each step justified)

In [10]:
def clean_insurance_data(df_raw):
    """
    1. Parse 'Effective To Date' as a real datetime.
    2. Add 'Response_Flag' (0/1) alongside the original 'Response' column.
    3. Do NOT treat Income=0 as missing -- verified above as a real pattern.
    4. No dedup/imputation needed -- verified zero duplicates, zero missing.
    5. Add 'Policy_Tier' extracted from 'Policy' (confirmed hierarchy).
    """
    df = df_raw.copy()
    df['Effective To Date'] = pd.to_datetime(df['Effective To Date'], format='%m/%d/%y')
    df['Response_Flag'] = (df['Response'] == 'Yes').astype(int)

    zero_income_employment = df.loc[df['Income'] == 0, 'EmploymentStatus'].unique()
    assert list(zero_income_employment) == ['Unemployed'], "Income=0 pattern changed -- re-check"

    df['Policy_Tier'] = df['Policy'].str.extract(r'(L\d)')
    return df

df = clean_insurance_data(df_raw)
print("Clean shape:", df.shape, "(no rows dropped)")
print(df[['Response', 'Response_Flag', 'Policy', 'Policy_Tier']].head())

Clean shape: (9134, 26) (no rows dropped)
  Response  Response_Flag        Policy Policy_Tier
0       No              0  Corporate L3          L3
1       No              0   Personal L3          L3
2       No              0   Personal L3          L3
3       No              0  Corporate L2          L2
4       No              0   Personal L1          L1


## Reusable analytical functions

In [11]:
def response_rate_by(df, group_col):
    """Response rate (%) and sample size by a grouping column."""
    return df.groupby(group_col, observed=True).agg(
        n=('Response_Flag', 'size'),
        response_rate_pct=('Response_Flag', lambda x: x.mean() * 100)
    ).sort_values('response_rate_pct', ascending=False)

def proxy_loss_ratio(df, group_col=None):
    """
    Proxy loss ratio = Total Claim Amount / (Monthly Premium Auto * 12)
    ASSUMPTION FLAG: directional estimate only, not an actuarial loss ratio.
    """
    df = df.copy()
    df['proxy_loss_ratio'] = df['Total Claim Amount'] / (df['Monthly Premium Auto'] * 12)
    if group_col:
        return df.groupby(group_col, observed=True)['proxy_loss_ratio'].agg(['mean','median','count']).sort_values('mean', ascending=False)
    return df['proxy_loss_ratio']

def flag_small_segments(summary_df, n_col='n', threshold=100):
    summary_df = summary_df.copy()
    summary_df['low_sample_warning'] = summary_df[n_col] < threshold
    return summary_df

print("Reusable functions defined.")

Reusable functions defined.


## Analysis D1 — Distribution shapes of key measures

**Question:** What do the distributions of CLV, Income, Monthly Premium, and
Total Claim Amount actually look like?

In [12]:
measures = ['Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Total Claim Amount']

print("--- Skewness (0 = symmetric, >1 = strongly right-skewed) ---")
for m in measures:
    skew = stats.skew(df[m])
    print(f"{m:28s} skew={skew:.2f}  mean={df[m].mean():,.0f}  median={df[m].median():,.0f}")

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, m in enumerate(measures):
    axes[0, i].hist(df[m], bins=40, color="#1F3864", alpha=0.8)
    axes[0, i].set_title(m, fontsize=10, fontweight='bold')
    axes[0, i].axvline(df[m].mean(), color='red', linestyle='--', linewidth=1, label='mean')
    axes[0, i].axvline(df[m].median(), color='orange', linestyle='--', linewidth=1, label='median')
    axes[0, i].legend(fontsize=7)
    axes[1, i].boxplot(df[m], vert=True)
    axes[1, i].set_xticklabels([''])
plt.suptitle("Distribution of Key Measures: Histogram (top) + Box Plot (bottom)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

--- Skewness (0 = symmetric, >1 = strongly right-skewed) ---
Customer Lifetime Value      skew=3.03  mean=8,005  median=5,780
Income                       skew=0.29  mean=37,657  median=33,890
Monthly Premium Auto         skew=2.12  mean=93  median=83
Total Claim Amount           skew=1.71  mean=434  median=384


/var/folders/tk/fdypsrw94kg40x6m2x674k_m0000gn/T/ipykernel_71563/4043922744.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**What this tells us:** All four measures are right-skewed (CLV skew=3.03 is
especially strong) — median is the safer statistic to report throughout this
project. The Income histogram shows a sharp spike at 0 (the Unemployed
segment) and an unusually flat spread from ~20,000-100,000 above that, which
is less noisy than typical real-world income data.

**What this does NOT tell us:** whether the "outlier" values are errors —
already checked in Part 1 and found to be real (luxury vehicles).

## Analysis D3 
#### — Response rate by offer type and channel

**Question:** Which renewal offers and sales channels actually convert?

In [13]:
print("--- Response rate by Renew Offer Type ---")
offer_summary = flag_small_segments(response_rate_by(df, 'Renew Offer Type'))
print(offer_summary)

print("\n--- Response rate by Sales Channel ---")
channel_summary = flag_small_segments(response_rate_by(df, 'Sales Channel'))
print(channel_summary)

print(f"\nOverall response rate: {df['Response_Flag'].mean()*100:.2f}% (n={len(df)})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
o = offer_summary.sort_index()
ax1.bar(o.index, o['response_rate_pct'], color="#E8794A")
ax1.set_title("Response Rate by Renew Offer Type", fontweight='bold')
ax1.set_ylabel("Response rate (%)")
for i, (idx, row) in enumerate(o.iterrows()):
    ax1.text(i, row['response_rate_pct'] + 0.5, f"n={row['n']}", ha='center', fontsize=9)

c = channel_summary.sort_index()
ax2.bar(c.index, c['response_rate_pct'], color="#5FD4D0")
ax2.set_title("Response Rate by Sales Channel", fontweight='bold')
ax2.set_ylabel("Response rate (%)")
for i, (idx, row) in enumerate(c.iterrows()):
    ax2.text(i, row['response_rate_pct'] + 0.5, f"n={row['n']}", ha='center', fontsize=9)
plt.tight_layout()
plt.show()

--- Response rate by Renew Offer Type ---
                     n  response_rate_pct  low_sample_warning
Renew Offer Type                                             
Offer2            2926          23.376623               False
Offer1            3752          15.831557               False
Offer3            1432           2.094972               False
Offer4            1024           0.000000               False

--- Response rate by Sales Channel ---
                  n  response_rate_pct  low_sample_warning
Sales Channel                                             
Agent          3477          19.154443               False
Web            1325          11.773585               False
Branch         2567          11.453058               False
Call Center    1765          10.878187               False

Overall response rate: 14.32% (n=9134)


/var/folders/tk/fdypsrw94kg40x6m2x674k_m0000gn/T/ipykernel_71563/4003133115.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**What this tells us:** Offer2 outperforms (23.4% vs. 14.3% baseline), while
Offer4 (n=1,024) produced **zero** positive responses. Agent channel (19.2%)
clearly outperforms Web/Branch/Call Center (~11% each).

**What this does NOT tell us:** *why* Offer4 failed completely, or whether
Agent's edge is genuine persuasion or a selection effect (agents may be
routed higher-intent customers). Both investigated next / flagged as
untested.

## Analysis 
#### — Diagnosing the Offer4 anomaly

**Question:** Is Offer4's 0% response concentrated in a particular segment,
or is it a uniform, structural pattern?

In [14]:
offer4 = df[df['Renew Offer Type'] == 'Offer4']
other = df[df['Renew Offer Type'] != 'Offer4']

print("--- Offer4 composition vs expectation (spread evenly?) ---")
for col in ['Sales Channel', 'Coverage', 'EmploymentStatus']:
    print(f"\n{col}:")
    print((offer4[col].value_counts(normalize=True) * 100).round(1))

print("\n--- Numeric profile: Offer4 vs other offers (medians) ---")
compare_cols = ['Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Months Since Policy Inception']
print(pd.DataFrame({'Offer4_median': offer4[compare_cols].median(),
                     'Other_offers_median': other[compare_cols].median()}))

print("\n--- Response rate by Offer x Channel (is Offer4 uniquely flat?) ---")
print(pd.crosstab(df['Renew Offer Type'], df['Sales Channel'], values=df['Response_Flag'], aggfunc='mean').round(3) * 100)

--- Offer4 composition vs expectation (spread evenly?) ---

Sales Channel:
Sales Channel
Agent          28.4
Branch         27.9
Call Center    23.3
Web            20.3
Name: proportion, dtype: float64

Coverage:
Coverage
Basic       63.3
Extended    30.5
Premium      6.2
Name: proportion, dtype: float64

EmploymentStatus:
EmploymentStatus
Employed         73.4
Unemployed       18.6
Medical Leave     3.5
Disabled          3.3
Retired           1.2
Name: proportion, dtype: float64

--- Numeric profile: Offer4 vs other offers (medians) ---
                               Offer4_median  Other_offers_median
Customer Lifetime Value          5257.221929          5872.282919
Income                          41452.500000         33190.000000
Monthly Premium Auto               76.000000            84.000000
Months Since Policy Inception      40.000000            49.000000

--- Response rate by Offer x Channel (is Offer4 uniquely flat?) ---
Sales Channel     Agent  Branch  Call Center   Web
Renew 

**What this tells us:** Offer4's customer mix is unremarkable and proportional
to the rest of the book across channel, state, and employment status, with
no outlying median profile. The key finding is in the last table: Offer4
shows **0% response across every single channel**, while Offer1/Offer2/Offer3
all show real channel-to-channel variation. Total flatness across many
subgroups (n=1,024) is a different kind of pattern than ordinary
underperformance.

**What this does NOT tell us:** whether this is a genuine business reality
(an unattractive offer) or a data/labeling artifact (e.g. a holdout/control
group). This needs to be resolved with whoever owns the source campaign data
— EDA alone cannot distinguish between these explanations.

## Analysis
#### — What actually drives Customer Lifetime Value?

**Question:** Why does CLV barely correlate with Income (r=0.02) despite
being the obvious assumption?

In [15]:
numeric_cols = ['Customer Lifetime Value', 'Income', 'Monthly Premium Auto',
                 'Total Claim Amount', 'Number of Policies', 'Months Since Policy Inception',
                 'Number of Open Complaints', 'Months Since Last Claim']
corr = df[numeric_cols].corr()
print("--- Correlation with CLV ---")
print(corr['Customer Lifetime Value'].sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im = axes[0].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(len(numeric_cols))); axes[0].set_xticklabels(numeric_cols, rotation=90, fontsize=8)
axes[0].set_yticks(range(len(numeric_cols))); axes[0].set_yticklabels(numeric_cols, fontsize=8)
axes[0].set_title("Correlation Heatmap", fontweight='bold')
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        axes[0].text(j, i, f"{corr.iloc[i,j]:.2f}", ha='center', va='center', fontsize=6)
plt.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].scatter(df['Monthly Premium Auto'], df['Customer Lifetime Value'], alpha=0.15, s=8, color="#1F3864")
axes[1].set_xlabel("Monthly Premium Auto"); axes[1].set_ylabel("Customer Lifetime Value")
axes[1].set_title("CLV vs Monthly Premium (r=0.40)", fontweight='bold')
plt.tight_layout()
plt.show()

--- Correlation with CLV ---
Customer Lifetime Value          1.000000
Monthly Premium Auto             0.396262
Total Claim Amount               0.226451
Income                           0.024366
Number of Policies               0.021955
Months Since Last Claim          0.011517
Months Since Policy Inception    0.009418
Number of Open Complaints       -0.036343
Name: Customer Lifetime Value, dtype: float64


/var/folders/tk/fdypsrw94kg40x6m2x674k_m0000gn/T/ipykernel_71563/1303055838.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**What this tells us:** only Premium (r=0.40) and Claim Amount (r=0.23) show
any real relationship with CLV — Income, Policy Count, tenure, and complaints
are all effectively uncorrelated (|r| < 0.04). The scatter plot shows
distinct straight-line "ray" patterns radiating from the origin, a visual
signature more consistent with a **formula-derived field** than organic
customer behavior.

**What this does NOT tell us:** the exact formula, or whether this dataset's
CLV matches how your actual stakeholders define customer value. Flagged as
a hypothesis needing confirmation, not a settled fact.

## Analysis Ex3 — Proxy loss ratio by segment

**Question:** Does claims cost relative to premium vary meaningfully by
vehicle class, state, or coverage tier?

In [16]:
print("--- Proxy loss ratio by Vehicle Class ---")
by_vehicle = proxy_loss_ratio(df, 'Vehicle Class')
print(by_vehicle)
print("\n--- by State ---")
print(proxy_loss_ratio(df, 'State'))
print("\n--- by Coverage ---")
print(proxy_loss_ratio(df, 'Coverage'))

fig, ax = plt.subplots(figsize=(9, 5.5))
bv = by_vehicle.sort_values('mean', ascending=True)
ax.barh(bv.index, bv['mean'], color="#E8794A")
ax.axvline(1.0, color='black', linestyle='--', linewidth=1, label='Break-even (ratio = 1.0)')
ax.set_xlabel("Proxy Loss Ratio (Total Claim Amount / Annualized Premium)")
ax.set_title("Proxy Loss Ratio by Vehicle Class\n(ESTIMATE -- not an actuarial loss ratio)", fontweight='bold', fontsize=11)
ax.legend()
for i, (idx, row) in enumerate(bv.iterrows()):
    ax.text(row['mean'] + 0.02, i, f"n={int(row['count'])}", va='center', fontsize=8)
plt.tight_layout()
plt.show()

--- Proxy loss ratio by Vehicle Class ---
                   mean  median  count
Vehicle Class                         
Luxury Car     0.444565     0.4    163
Luxury SUV     0.426056     0.4    184
SUV            0.393875     0.4   1796
Sports Car     0.380920     0.4    484
Two-Door Car   0.379588     0.4   1886
Four-Door Car  0.377480     0.4   4621

--- by State ---
                mean  median  count
State                              
Washington  0.389718     0.4    798
California  0.384974     0.4   3150
Nevada      0.383471     0.4    882
Arizona     0.381924     0.4   1703
Oregon      0.380838     0.4   2601

--- by Coverage ---
              mean  median  count
Coverage                         
Premium   0.402318     0.4    824
Extended  0.382877     0.4   2742
Basic     0.381016     0.4   5568


/var/folders/tk/fdypsrw94kg40x6m2x674k_m0000gn/T/ipykernel_71563/890028411.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**What this tells us:** the median proxy loss ratio is **exactly 0.4 in every
single segment checked** (6 vehicle classes, 5 states, 3 coverage tiers),
with means clustering tightly (0.377-0.445). Real claims data does not
typically produce this little variation across genuinely different risk
groups — this reinforces the Dx2 finding that `Total Claim Amount` is likely
linked to premium by a near-fixed formula.

**What this does NOT tell us:** which vehicle classes or states are actually
riskier. **Practical implication: this dataset cannot support segment-level
risk/pricing analysis** — presenting small differences here (e.g. Luxury Car
0.44 vs Four-Door 0.38) as a real risk signal would be reporting noise as
insight.

Observed Facts vs. Interpretations vs. Untested Hypotheses

| Category | Example |
|---|---|
| **Observed fact** | Offer4 = 0/1,024 responses, flat across every channel. CLV-Income r=0.02. Proxy loss ratio median=0.4 in every segment checked. Agent channel response rate=19.2% vs ~11% elsewhere. |
| **Reasonable interpretation** | CLV and Claim Amount both show signs of formulaic derivation. Offer4's uniform zero suggests something structurally different from ordinary underperformance. |
| **Untested hypothesis** | Offer4 was a holdout/control group. Agent channel is causally more effective (vs. merely correlated via customer selection). CLV = a specific formula involving Premium, Policy Count, and Tenure. |

**No correlation found in this analysis is treated as causation.** Every "what explains this" question above is paired with at least one alternative explanation the data cannot rule out.

## Key Findings Summary

1. **Offer4 (n=1,024) shows 0% response across every sales channel** — flagged for business investigation, not treated as "a weak offer."
2. **CLV is driven by Premium (r=0.40), not Income (r=0.02)** — contradicts the common assumption that income predicts customer value.
3. **The dataset cannot support segment-level claims/risk analysis** — proxy loss ratio shows almost no variation across vehicle class, state, or coverage.
4. **Agent channel outperforms other channels (19.2% vs ~11%)** — real, but causation vs. selection effect remains unresolved.
5. **Core financial measures are heavily right-skewed** — median, not mean, is the correct default statistic for this dataset.